# Semantic Chunking Experiment

## Objective

The objective of this experiment is to develop and evaluate a semantic chunking strategy for the RAG pipeline using the structured document tree produced from Markdown.

The previous chunking pipeline operated directly on extracted text and relied primarily on `RecursiveCharacterTextSplitter`. The new approach uses the document's structural hierarchy as the primary semantic signal before applying size-based splitting.

---


## Current Document Representation

The document is now represented through the following pipeline:

```text
PDF
 ↓
Markdown Extraction
 ↓
Markdown Tree
 ↓
Semantic Chunking
 ↓
Final Chunks
```

The Markdown tree preserves document hierarchy such as:

* Headings
* Sub-headings
* Paragraphs
* Lists
* Tables
* Nested sections

This structure will be used to identify meaningful semantic units.

---


In [ ]:
"""
Step 0
reading the file and preprocess it to have the tree
"""
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))


import pymupdf
import pymupdf4llm
from src.preprocessing.ingestion_preprocessing import HeaderInfoBuilder
FILE_PATH ="../src/pdfs/logical_reasoning.pdf"


header_builder = HeaderInfoBuilder()
hdr_info = header_builder.build(FILE_PATH)


pymupdf4llm.use_layout(False)

doc = pymupdf.open(FILE_PATH)

markdown = pymupdf4llm.to_markdown(
    doc,
    show_progress=True,
    # use_ocr=False,
    hdr_info=hdr_info
)


# Markdown -> Tree
from src.preprocessing.markdown_parser import build_tree

markdown_tree = build_tree(text=markdown)

In [ ]:
markdown_tree

In [ ]:
from anytree import Node, RenderTree


def build_anytree(tree, parent=None):
    node = Node(tree["text"], parent=parent)

    for child in tree.get("children", []):
        build_anytree(child, parent=node)

    return node


root = build_anytree(markdown_tree)

for pre, _, node in RenderTree(root):
    print(f"{pre}{node.name}")



# 1. Semantic Chunking

In [ ]:
# Traverse the Tree
def traverse_tree(tree)-> list[dict]:
    path_stack = []
    chunks = []

    for node in tree:

        # Building the Path Stack
        path_stack.append(node["text"])

        has_children = bool(node["children"])

        if node.get("content") is not None:

            # Produce breadcrumb
            bread_crumb = path_stack.copy()

            content = node.get("content", "")

            # Node-level content
            if has_children:

                subtopics = [
                    child["text"]
                    for child in node["children"]
                ]

                chunk_content = (
                    f"[Content]: {content}\n"
                    f"[Subtopics]: {subtopics}"
                )

            # Leaf-level content
            else:
                chunk_content = content

            # Create chunk
            chunks.append({
                "title": node.get("text", ""),
                "content": chunk_content,
                "breadcrumb": bread_crumb
            })

        # Traverse children
        if has_children:
            traverse_tree(node["children"])

        # Leave current node
        path_stack.pop()
    return chunks

In [ ]:
chunks = traverse_tree([markdown_tree])

In [ ]:
chunks[2:]

## 2. Size Controlled Chunk Split

In [ ]:
from src.chunking.chunker import text_splitter

In [ ]:
final_chunks = []
for chunk in chunks:
   splitted_chunks =  text_splitter(chunk['content'])

   if len(splitted_chunks) >1:
      print(len(splitted_chunks))

   final_chunks.append({
      'title': chunk['title'],
      'chunks': splitted_chunks,
      'breadcrumb': chunk['breadcrumb']
   })

In [ ]:
final_chunks[7:10]